# 01 · Coleta e preparação dos dados

**Autor:** Igor Seberino · **Fase CRISP-DM:** entendimento e preparação dos dados

Objetivo da etapa: obter o dataset AI4I 2020 de forma reprodutível, verificar a qualidade
dos dados, documentar as inconsistências de rótulo e entregar a todos os notebooks seguintes
o mesmo DataFrame analítico (colunas renomeadas, tipadas e com atributos físicos derivados)
e a mesma divisão treino/teste.

In [1]:
# Ambiente: coloca src/ no sys.path (o notebook roda a partir de notebooks/ ou da raiz)
import sys
from pathlib import Path

RAIZ = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "src"))

In [2]:
import pandas as pd

from manutencao.config import (
    CAMINHO_CSV,
    COL_ALVO,
    COLS_ENGENHARIA,
    COLS_MODOS,
    COLS_VAZAMENTO,
    MAPA_COLUNAS,
    MODOS_FALHA,
    SHA256_CSV,
)
from manutencao.dados import carregar_bruto, inconsistencias_rotulo, sha256_arquivo, validar
from manutencao.preparacao import dividir_treino_teste, preparar
from manutencao.util import configurar_graficos, salvar_tabela

configurar_graficos()
pd.set_option("display.max_columns", 30)

## 1. Fonte

- **Base:** *AI4I 2020 Predictive Maintenance Dataset*, UCI Machine Learning Repository,
  id 601 (UCI MACHINE LEARNING REPOSITORY, 2020), DOI
  [10.24432/C5HS5C](https://doi.org/10.24432/C5HS5C), licença CC BY 4.0. O conjunto foi
  publicado com o artigo de Matzka (2020).
- **Natureza:** dataset **sintético** que reproduz dados reais de manutenção de uma máquina de
  usinagem: cada linha é um ciclo de processo com leituras de sensores e o rótulo de falha.
- **Reprodutibilidade:** o CSV original está versionado em `data/raw/ai4i2020.csv`.
  Se o arquivo faltar, `carregar_bruto()` chama `baixar_csv()`, que baixa o zip oficial da UCI
  e confere o SHA-256 antes de aceitar o arquivo. Aqui conferimos que o arquivo lido é
  exatamente o oficial.

In [3]:
df_bruto = carregar_bruto()
print("linhas × colunas:", df_bruto.shape)
print("SHA-256 confere com o oficial:", sha256_arquivo(CAMINHO_CSV) == SHA256_CSV)
print(f"tamanho do arquivo: {CAMINHO_CSV.stat().st_size:,} bytes".replace(",", "."))
df_bruto.head()

linhas × colunas: (10000, 14)
SHA-256 confere com o oficial: True
tamanho do arquivo: 522.048 bytes


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
df_bruto.dtypes.to_frame("tipo_pandas")

,tipo_pandas
UDI,int64
Product ID,str
Type,str
Air temperature [K],float64
Process temperature [K],float64
Rotational speed [rpm],int64
Torque [Nm],float64
Tool wear [min],int64
Machine failure,int64
TWF,int64


## 2. Validação de qualidade

Antes de qualquer análise, `validar()` executa 14 verificações e devolve o esperado e o obtido
de cada uma:

- **estrutura:** colunas esperadas e 10.000 linhas;
- **completude e unicidade:** nenhum valor ausente, nenhuma linha duplicada, `UDI` e
  `Product ID` únicos;
- **coerência de domínio:** `Type` só em {L, M, H} e igual à 1ª letra do `Product ID`;
  temperaturas na faixa física de 290 a 320 K; temperatura de processo maior que a do ar;
  rotação positiva; torque e desgaste não negativos;
- **rótulos:** alvo e modos de falha só com 0 ou 1.

O `assert` faz o notebook (e o CI) falhar se alguma verificação reprovar.

In [5]:
qualidade = validar(df_bruto)
salvar_tabela(qualidade, "01_validacao_qualidade")
assert qualidade["ok"].all()
qualidade

,verificacao,esperado,obtido,ok
0,colunas esperadas,"['desgaste_ferramenta_min', 'falha', 'falha_hd...","['desgaste_ferramenta_min', 'falha', 'falha_hd...",True
1,quantidade de linhas,10000,10000,True
2,valores ausentes,0,0,True
3,linhas duplicadas,0,0,True
4,udi único,True,True,True
5,id_produto único,True,True,True
6,"tipo em {L, M, H}",True,True,True
7,1ª letra do id_produto = tipo,True,True,True
8,temperaturas entre 290 e 320 K,True,True,True
9,temp_processo > temp_ar,True,True,True


## 3. Inconsistências de rótulo

Pela documentação, `Machine failure = 1` quando **pelo menos um** dos cinco modos de falha
(TWF, HDF, PWF, OSF, RNF) ocorre. O arquivo não segue isso em todas as linhas:

- **falha sem modo:** ciclos com `Machine failure = 1` e nenhum modo marcado;
- **modo sem falha:** ciclos com um modo marcado (todos RNF, falha aleatória) e
  `Machine failure = 0`.

A página da UCI também diz que RNF ocorre em 5 ciclos, mas o arquivo tem 19 ciclos com
`RNF = 1`. Reportamos o que o arquivo tem.

**Decisão D5:** o alvo é `Machine failure` **como publicado**, sem re-rotular essas linhas.
Motivo: mantém o resultado comparável com os trabalhos que usam a mesma base. As linhas
inconsistentes são tratadas como ruído de rótulo e ficam registradas em
`reports/tables/01_inconsistencias_rotulo.csv`.

In [6]:
inc = inconsistencias_rotulo(df_bruto)
salvar_tabela(inc, "01_inconsistencias_rotulo")
modos_inc = (
    inc[inc["inconsistencia"] == "modo sem falha"][COLS_MODOS].sum().loc[lambda s: s > 0].to_dict()
)
print("ciclos com RNF = 1 no arquivo:", int(df_bruto["RNF"].sum()))
print("modos marcados nas linhas 'modo sem falha':", modos_inc)
inc["inconsistencia"].value_counts()

ciclos com RNF = 1 no arquivo: 19
modos marcados nas linhas 'modo sem falha': {'falha_rnf': 18}


inconsistencia
modo sem falha    18
falha sem modo     9
Name: count, dtype: int64

In [7]:
inc.head(10)

,udi,id_produto,tipo,temp_ar_k,temp_processo_k,rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf,inconsistencia
1221,1222,M16081,M,297.0,308.3,1399,46.4,132,0,0,0,0,0,1,modo sem falha
1302,1303,L48482,L,298.6,309.8,1505,45.7,144,0,0,0,0,0,1,modo sem falha
1437,1438,H30851,H,298.8,309.9,1439,45.2,40,1,0,0,0,0,0,falha sem modo
1748,1749,H31162,H,298.4,307.7,1626,31.1,166,0,0,0,0,0,1,modo sem falha
2072,2073,L49252,L,299.6,309.5,1570,35.5,189,0,0,0,0,0,1,modo sem falha
2559,2560,L49739,L,299.3,309.0,1447,50.4,140,0,0,0,0,0,1,modo sem falha
2749,2750,M17609,M,299.7,309.2,1685,28.9,179,1,0,0,0,0,0,falha sem modo
3065,3066,M17925,M,300.1,309.2,1687,27.7,95,0,0,0,0,0,1,modo sem falha
3452,3453,H32866,H,301.6,310.5,1602,32.3,2,0,0,0,0,0,1,modo sem falha
4044,4045,M18904,M,301.9,310.9,1419,47.7,20,1,0,0,0,0,0,falha sem modo


## 4. Renomeação e tipagem

Os nomes originais têm espaços, unidades entre colchetes e estão em inglês. `preparar()` troca
para nomes em português, sem espaços e com a unidade no sufixo (tabela abaixo, de
`MAPA_COLUNAS`).

`tipo` vira **categórica ordenada L < M < H**: o tipo indica a variante de qualidade do produto
(baixa, média, alta), então a ordem tem significado e entra nos modelos e nos gráficos sempre
na mesma sequência. Contagens e sinais binários ficam inteiros; temperaturas e torque ficam
reais.

In [8]:
pd.DataFrame(list(MAPA_COLUNAS.items()), columns=["coluna_original", "coluna_nova"])

,coluna_original,coluna_nova
0,UDI,udi
1,Product ID,id_produto
2,Type,tipo
3,Air temperature [K],temp_ar_k
4,Process temperature [K],temp_processo_k
5,Rotational speed [rpm],rotacao_rpm
6,Torque [Nm],torque_nm
7,Tool wear [min],desgaste_ferramenta_min
8,Machine failure,falha
9,TWF,falha_twf


In [9]:
df = preparar(df_bruto)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   udi                        10000 non-null  int64   
 1   id_produto                 10000 non-null  str     
 2   tipo                       10000 non-null  category
 3   temp_ar_k                  10000 non-null  float64 
 4   temp_processo_k            10000 non-null  float64 
 5   rotacao_rpm                10000 non-null  int64   
 6   torque_nm                  10000 non-null  float64 
 7   desgaste_ferramenta_min    10000 non-null  int64   
 8   falha                      10000 non-null  int64   
 9   falha_twf                  10000 non-null  int64   
 10  falha_hdf                  10000 non-null  int64   
 11  falha_pwf                  10000 non-null  int64   
 12  falha_osf                  10000 non-null  int64   
 13  falha_rnf                  10000 non-null  

In [10]:
df.head()

,udi,id_produto,tipo,temp_ar_k,temp_processo_k,rotacao_rpm,torque_nm,desgaste_ferramenta_min,falha,falha_twf,falha_hdf,falha_pwf,falha_osf,falha_rnf,dif_temperatura_k,potencia_w,esforco_ferramenta_min_nm
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,6951.590560,0.0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,6826.722724,138.9
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,7749.387543,247.0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,5927.504659,276.5
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,5897.816608,360.0


## 5. Atributos físicos

Três atributos derivados da física da usinagem, o conhecimento que um engenheiro de processo
usaria para olhar a máquina:

- **Diferença de temperatura** (dissipação de calor):
  $\Delta T = T_{proc} - T_{ar}$, em K. Diferença pequena significa que o calor do processo não
  está sendo dissipado.
- **Potência mecânica**: $P = \tau \cdot \omega = \tau \cdot n \cdot 2\pi/60$, em W, com
  $\tau$ o torque (N·m) e $n$ a rotação (rpm). Potência baixa ou alta demais indica falha de
  potência.
- **Esforço da ferramenta** (sobrecarga): $E = \text{desgaste} \cdot \tau$, em min·N·m. Uma
  ferramenta gasta sob torque alto sofre sobrecarga.

A documentação da UCI descreve as regras que geraram três dos modos de falha justamente com
essas grandezas. A célula abaixo aplica cada regra aos atributos derivados e compara com o
rótulo do modo.

In [11]:
regras = {
    "HDF": (df["dif_temperatura_k"] < 8.6) & (df["rotacao_rpm"] < 1380),
    "PWF": (df["potencia_w"] < 3500) | (df["potencia_w"] > 9000),
    "OSF": df["esforco_ferramenta_min_nm"]
    > df["tipo"].map({"L": 11000, "M": 12000, "H": 13000}).astype(float),
}
regras_vs_rotulos = pd.DataFrame(
    [
        {
            "modo": modo,
            "descricao": MODOS_FALHA[modo][1],
            "ciclos_pela_regra": int(mascara.sum()),
            "ciclos_com_rotulo": int(df[MODOS_FALHA[modo][0]].sum()),
            "coincidem": bool((mascara == (df[MODOS_FALHA[modo][0]] == 1)).all()),
        }
        for modo, mascara in regras.items()
    ]
)
salvar_tabela(regras_vs_rotulos, "01_regras_vs_rotulos")
regras_vs_rotulos

,modo,descricao,ciclos_pela_regra,ciclos_com_rotulo,coincidem
0,HDF,Falha de dissipação de calor,115,115,True
1,PWF,Falha de potência,95,95,True
2,OSF,Sobrecarga (overstrain),98,98,True


In [12]:
falhas = df[df[COL_ALVO] == 1]
deterministicas = falhas[["falha_hdf", "falha_pwf", "falha_osf"]].any(axis=1)
print("falhas no total:", len(falhas))
print("falhas com HDF, PWF ou OSF (explicadas por regra):", int(deterministicas.sum()))
print("falhas restantes (só TWF, RNF ou sem modo):", int((~deterministicas).sum()))

falhas no total: 339
falhas com HDF, PWF ou OSF (explicadas por regra): 287
falhas restantes (só TWF, RNF ou sem modo): 52


**Interpretação.** As três regras reproduzem os rótulos em 100 % dos ciclos. Isso significa que
a maior parte das falhas é **determinística**: é uma função das variáveis medidas, e os
atributos derivados expõem essa função diretamente. As falhas restantes (desgaste da
ferramenta, falha aleatória e as sem modo) não seguem uma regra sobre a leitura de um ciclo.

Consequências:

- **para a modelagem (notebook 04):** é de esperar desempenho muito alto nos modos HDF, PWF e
  OSF, sobretudo com os atributos de engenharia; o erro que sobra se concentra nas falhas não
  determinísticas, que funcionam como um teto para o recall;
- **para a discussão (§3.3):** o dataset é sintético, gerado por essas regras. O desempenho não
  se transfere automaticamente para uma fábrica real, em que as falhas não seguem limiares
  exatos.

## 6. Dicionário de dados

As 17 colunas do DataFrame analítico: 14 originais e 3 derivadas. O papel indica como cada
coluna pode ser usada; identificadores e modos de falha são **proibidos como atributo**
(decisão D6) e `separar_x_y()` recusa essas colunas.

In [13]:
descricoes = {
    "udi": ("identificador", "-", "Identificador único do ciclo (1 a 10.000)"),
    "id_produto": (
        "identificador",
        "-",
        "Identificador do produto: letra do tipo + número de série",
    ),
    "tipo": ("atributo", "-", "Variante de qualidade do produto: L (baixa), M (média), H (alta)"),
    "temp_ar_k": ("atributo", "K", "Temperatura do ar ambiente"),
    "temp_processo_k": ("atributo", "K", "Temperatura do processo"),
    "rotacao_rpm": ("atributo", "rpm", "Velocidade de rotação do eixo"),
    "torque_nm": ("atributo", "N·m", "Torque aplicado"),
    "desgaste_ferramenta_min": ("atributo", "min", "Tempo de uso acumulado da ferramenta"),
    "falha": ("alvo", "-", "1 se o ciclo terminou em falha da máquina"),
    "dif_temperatura_k": ("atributo", "K", "temp_processo_k − temp_ar_k (dissipação de calor)"),
    "potencia_w": ("atributo", "W", "torque_nm × rotacao_rpm × 2π/60 (potência mecânica)"),
    "esforco_ferramenta_min_nm": (
        "atributo",
        "min·N·m",
        "desgaste_ferramenta_min × torque_nm (sobrecarga da ferramenta)",
    ),
}
for sigla, (coluna, nome) in MODOS_FALHA.items():
    descricoes[coluna] = (
        "modo de falha — proibido como atributo",
        "-",
        f"1 se ocorreu o modo {sigla}: {nome.lower()}",
    )
for coluna in COLS_VAZAMENTO:
    papel, unidade, texto = descricoes[coluna]
    if papel == "identificador":
        descricoes[coluna] = ("identificador — proibido como atributo", unidade, texto)

dicionario = pd.DataFrame(
    [
        {
            "coluna": coluna,
            "origem": "derivada" if coluna in COLS_ENGENHARIA else "original",
            "tipo": str(df[coluna].dtype),
            "unidade": descricoes[coluna][1],
            "papel": descricoes[coluna][0],
            "descricao": descricoes[coluna][2],
        }
        for coluna in df.columns
    ]
)
assert len(dicionario) == 17
salvar_tabela(dicionario, "01_dicionario_dados")
dicionario

,coluna,origem,tipo,unidade,papel,descricao
0,udi,original,int64,-,identificador — proibido como atributo,Identificador único do ciclo (1 a 10.000)
1,id_produto,original,str,-,identificador — proibido como atributo,Identificador do produto: letra do tipo + núme...
2,tipo,original,category,-,atributo,"Variante de qualidade do produto: L (baixa), M..."
3,temp_ar_k,original,float64,K,atributo,Temperatura do ar ambiente
4,temp_processo_k,original,float64,K,atributo,Temperatura do processo
5,rotacao_rpm,original,int64,rpm,atributo,Velocidade de rotação do eixo
6,torque_nm,original,float64,N·m,atributo,Torque aplicado
7,desgaste_ferramenta_min,original,int64,min,atributo,Tempo de uso acumulado da ferramenta
8,falha,original,int64,-,alvo,1 se o ciclo terminou em falha da máquina
9,falha_twf,original,int64,-,modo de falha — proibido como atributo,1 se ocorreu o modo TWF: desgaste da ferramenta


## 7. Divisão treino/teste

Divisão **única no projeto**, feita só por `dividir_treino_teste()`: 80 % treino e 20 % teste,
estratificada pelo alvo (as duas partes mantêm a mesma proporção de falhas) e com `SEED = 42`.
Todos os notebooks usam a mesma divisão; o conjunto de teste fica reservado e só é usado no
fim do notebook 04 e no notebook 05.

In [14]:
treino, teste = dividir_treino_teste(df)
divisao = pd.DataFrame(
    [
        {
            "conjunto": nome,
            "ciclos": len(parte),
            "falhas": int(parte[COL_ALVO].sum()),
            "prevalencia_pct": round(100 * parte[COL_ALVO].mean(), 2),
        }
        for nome, parte in [("treino", treino), ("teste", teste), ("total", df)]
    ]
)
salvar_tabela(divisao, "01_divisao_treino_teste")
divisao

,conjunto,ciclos,falhas,prevalencia_pct
0,treino,8000,271,3.39
1,teste,2000,68,3.40
2,total,10000,339,3.39


In [15]:
df.groupby("tipo", observed=True)[COL_ALVO].agg(ciclos="size", falhas="sum").assign(
    taxa_falha_pct=lambda t: (100 * t["falhas"] / t["ciclos"]).round(2)
)

,ciclos,falhas,taxa_falha_pct
tipo,,,
L,6000,235,3.92
M,2997,83,2.77
H,1003,21,2.09


## Conclusões da etapa

- O arquivo lido é o oficial da UCI (SHA-256 conferido): **10.000 ciclos × 14 colunas**.
- As **14 verificações de qualidade** passaram: nenhum valor ausente, nenhuma duplicata,
  identificadores únicos, valores dentro das faixas físicas.
- Há **27 inconsistências de rótulo** (9 "falha sem modo" e 18 "modo sem falha", todas RNF);
  pela decisão D5 o rótulo publicado é mantido e as linhas ficam documentadas.
- O alvo é **desbalanceado**: 339 falhas em 10.000 ciclos (3,39 %); a taxa cai de L (3,92 %)
  para M (2,77 %) e H (2,09 %).
- Os três atributos físicos reproduzem **100 %** das regras de HDF, PWF e OSF, que respondem
  por **287 das 339 falhas**; as outras 52 não são determinísticas e limitam o recall possível.
- A divisão única tem **8.000 ciclos de treino e 2.000 de teste**, com 271 e 68 falhas
  (3,39 % e 3,40 %).
- Saídas em `reports/tables/01_*.csv`: validação, inconsistências, regras × rótulos,
  dicionário de dados e divisão.